# Semantic Correspondence — Colab (end-to-end)

This notebook runs **end-to-end on Google Colab**:

1. clone the repository
2. download + extract **SPair-71k** to `data/SPair-71k/`
3. install the project (`pip install -e ".[notebook]"`)
4. download pretrained weights to `checkpoints/`
5. write `config.yaml` and run `scripts/run_pipeline.py --config config.yaml`

**Artifacts** (logs, state, exports, checkpoints, weights) are stored under `runs/` and `checkpoints/`. This notebook can symlink them to Google Drive (`MyDrive/Colab Notebooks/AML_results/`) so resume works across restarts.


### 1. Runtime + CUDA sanity checks

Make sure you selected a **GPU runtime** (e.g. **T4 GPU**) in Colab: `Runtime → Change runtime type → Hardware accelerator → GPU`.


In [ ]:
# Ensure PyTorch can use CUDA (GPU runtime required).

import sys
import subprocess

# Check torch without importing it in this kernel first.
result = subprocess.run(
    [
        sys.executable,
        "-c",
        "import torch;\nprint(torch.__version__);\nprint('cuda_available=', torch.cuda.is_available());\nprint('torch_cuda=', torch.version.cuda)",
    ],
    capture_output=True,
    text=True,
)
print(result.stdout)

need_reinstall = ("cuda_available= False" in result.stdout) and ("torch_cuda= None" in result.stdout)

if need_reinstall:
    print("Reinstalling CUDA-enabled torch/torchvision...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--force-reinstall",
            "torch",
            "torchvision",
            "--index-url",
            "https://download.pytorch.org/whl/cu121",
        ],
        check=True,
    )

import torch

print("torch", torch.__version__)
print("cuda_available=", torch.cuda.is_available())
print("torch_cuda=", torch.version.cuda)
if torch.cuda.is_available():
    print("gpu=", torch.cuda.get_device_name(0))


### 2. (Optional) Cleanup old Colab workspace

If you rerun the notebook, you may have an old repo/dataset under `/content`. This step resets the ephemeral Colab filesystem (it does **not** delete anything on Google Drive).


In [ ]:
import shutil
from pathlib import Path

# If you rerun the notebook, you may have an old repo/dataset under /content.
# This does NOT delete anything on Google Drive.
FORCE_CLEAN = True

paths_to_remove = [
    Path("/content/Semantic-Correspondence"),
    Path("/content/sample_data"),
]

if FORCE_CLEAN:
    for p in paths_to_remove:
        if p.exists():
            print("Removing:", p)
            shutil.rmtree(p, ignore_errors=True)
else:
    print("FORCE_CLEAN=False: skipping cleanup")


### 3. Clone repository


### 4. (Recommended) Persist outputs to Google Drive

Colab storage under `/content` is ephemeral. To support **resume across sessions**, we store `runs/` and `checkpoints/` on Drive at:

- `MyDrive/Colab Notebooks/AML_results/`

Run the **next** cell to clone the repo, then run the Drive cell to create symlinks.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/Semantic-Correspondence')
REPO_URL = 'https://github.com/lucaosti/Semantic-Correspondence.git'

if not REPO_DIR.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('REPO =', REPO_DIR)


In [ ]:
# Drive persistence (see markdown cell above)

import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_DIR = Path('/content/Semantic-Correspondence')
BASE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/AML_results')
RUNS_DIR = BASE_DIR / 'runs'
CKPT_DIR = BASE_DIR / 'checkpoints'

RUNS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Create/refresh symlinks in the repo
os.chdir(REPO_DIR)
for link_name, target in [('runs', RUNS_DIR), ('checkpoints', CKPT_DIR)]:
    p = REPO_DIR / link_name
    if p.is_symlink() or p.is_file():
        p.unlink()
    elif p.is_dir():
        shutil.rmtree(p)
    os.symlink(str(target), str(p))
    print(f"Linked {p} -> {target}")


### 5. Download + extract SPair-71k


In [ ]:
import os
import tarfile
import urllib.request
from pathlib import Path

REPO_DIR = Path('/content/Semantic-Correspondence')
os.chdir(REPO_DIR)

SPAIR_URL = 'https://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz'
DATA_DIR = REPO_DIR / 'data'
SPAIR_ROOT = DATA_DIR / 'SPair-71k'
TAR_PATH = DATA_DIR / 'SPair-71k.tar.gz'

DATA_DIR.mkdir(parents=True, exist_ok=True)

if not SPAIR_ROOT.is_dir():
    if not TAR_PATH.is_file():
        print('Downloading:', SPAIR_URL)
        urllib.request.urlretrieve(SPAIR_URL, TAR_PATH)
        print('Saved:', TAR_PATH)
    else:
        print('Already present:', TAR_PATH)

    print('Extracting to:', DATA_DIR)
    with tarfile.open(TAR_PATH, 'r:gz') as tar:
        tar.extractall(path=DATA_DIR)

print('SPAIR_ROOT =', SPAIR_ROOT, '| exists =', SPAIR_ROOT.is_dir())


### 6. Install project (editable + notebook extras)


In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path('/content/Semantic-Correspondence')
os.chdir(REPO_DIR)

!{sys.executable} -m pip install -q -e ".[notebook]"

os.environ['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
print('OK:', REPO_DIR)


### 7. Download pretrained weights (DINOv2, DINOv3, SAM)


In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path('/content/Semantic-Correspondence')
os.chdir(REPO_DIR)

!{sys.executable} scripts/download_pretrained_weights.py


### 8. Write config.yaml (Colab defaults)


In [ ]:
import os
import sys
from pathlib import Path

import torch
import yaml

REPO_DIR = Path('/content/Semantic-Correspondence')
os.chdir(REPO_DIR)

cfg_path = REPO_DIR / 'config.yaml'

cfg = {
    'dataset': {
        'backend': 'sd4match',
        'metrics_backend': 'sd4match',
    },
    'paths': {
        'repo_root': str(REPO_DIR),
        'spair_root': str(REPO_DIR / 'data' / 'SPair-71k'),
        'checkpoint_dir': 'checkpoints',
        # Stored under `runs/` (symlinked to Drive if you enabled the Drive cell above).
        'output_dir': str(REPO_DIR / 'runs' / 'notebook_exports'),
    },
    'runtime': {
        # Use CUDA only if PyTorch reports it is available.
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'num_workers': 2,
        'preprocess': 'FIXED_RESIZE',
        'image_height': 784,
        'image_width': 784,
        'limit_pairs': 0,
        'eval_split': 'test',
        'alphas': [0.05, 0.1, 0.2],
        'wsa_window': 5,
        'wsa_temperature': 1.0,
        'log_batch_interval': 100,
    },
    'finetune': {
        'last_blocks': 2,
        'epochs': 100,
        'patience': 10,
        'batch_size': 10,
        'dinov2_weights': str(REPO_DIR / 'checkpoints' / 'dinov2_vitb14_pretrain.pth'),
        'dinov3_weights': str(REPO_DIR / 'checkpoints' / 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth'),
        'sam_checkpoint': str(REPO_DIR / 'checkpoints' / 'sam_vit_b_01ec64.pth'),
    },
    'lora': {
        'rank': 16,
        'epochs': 100,
        'patience': 3,
        'batch_size': 10,
    },
    'workflow_toggles': {
        'run_verify_dataset': True,
        'train_finetune': [True, True, True],
        'train_lora': [False, False, False],
        'run_eval_baseline': [True, True, True],
        'run_eval_baseline_wsa': [False, False, False],
        'run_eval_finetuned_checkpoint': [True, True, True],
        'run_eval_lora_checkpoint': [False, False, False],
        'run_export_metrics_tables': True,
        'run_pytest': False,
        'print_jupyter_notebook_hint': True,
        'pipeline_resume': True,
    },
}

with open(cfg_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True, default_flow_style=False)
print('Written:', cfg_path)


### 9. Run pipeline


In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path('/content/Semantic-Correspondence')
os.chdir(REPO_DIR)

!{sys.executable} scripts/run_pipeline.py --config config.yaml
